In [ ]:
%run Setting_Env_Variables_p2.py
google_project_id = %env GOOGLE_CLOUD_PROJECT
bucket = os.getenv("WORKSPACE_BUCKET")

import os
import subprocess
import pandas as pd
from datetime import datetime

import hail as hl
hl.init(gcs_requester_pays_configuration=google_project_id) #, log=f'{bucket}/hail_logs')
hl.default_reference(new_default_reference = "GRCh38")


In [ ]:
auxiliary_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux"
print(f'aux path: {auxiliary_path}')

vat_path = f'{auxiliary_path}/vat/*.gz'
print(f'vat path: {vat_path}')

clinvar_ht_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/multiMT/hail.mt'
print(f'clinvar ht path: {clinvar_ht_path}')

vds_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/vds/hail.vds'
print(f'vds path: {vds_path}')

flagged_samples_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
print(f'flagged samples path: {flagged_samples_path}')


# mt_wgs_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"
# flagged_samples = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
# array_path = "gs://vwb-aou-datasets-controlled/v9/microarray/hail.mt"
# # this path is in the official ct v8 fw
# bed_path = "gs://aou-tutorial-notebooks-wb-sunny-radish-6214/genomic_test_data/random_sites_GRCh38.txt"

parse by demographic

Get all LMNA associated variants for control groups

In [ ]:
%%time
types = {
    'clinvar_classification': hl.tarray(hl.tstr),
    'position': hl.tint,
    'omim_phenotypes_id': hl.tarray(hl.tint),
    'omim_phenotypes_name': hl.tarray(hl.tstr),
    'clinvar_phenotype': hl.tarray(hl.tstr),
}

vat_table = hl.import_table(
    vat_path, force=True, quote='"', delimiter='\t', force_bgz=True, types=types,
)
vat_table = vat_table.add_index(name='id')


In [ ]:
gene = 'LMNA'
ens_id = 'ENSG00000160789'

filt_vat = vat_table.filter(vat_table['gene_id'] == ens_id)

Get variants within 1 Mb of LMNA (genomic window; includes non-LMNA-annotated sites)

In [ ]:
# Shared LMNA ±1 Mb window (also used for VDS filtering below)
LMNA_START, LMNA_END = 156_082_572, 156_140_081
FLANK_BP = 1_000_000
window_start = LMNA_START - FLANK_BP
window_end = LMNA_END + FLANK_BP
print(f'chr1:{window_start}-{window_end}')

vat_1mb = vat_table.filter(
    (vat_table.contig == 'chr1')
    & (vat_table.position >= window_start)
    & (vat_table.position <= window_end)
)

## VDS: filter to LMNA ±1 Mb

In [ ]:
%%time
vds = hl.vds.read_vds(vds_path)

In [ ]:
# only work with data from chromosome 1 (location of LMNA)
vds = hl.vds.filter_chromosomes(vds, keep= ["chr1"])

In [ ]:
# Reuse LMNA ±1 Mb window defined above for VAT filtering
locus_interval = hl.parse_locus_interval(
    f'chr1:{window_start}-{window_end}',
    reference_genome='GRCh38',
)
print(locus_interval)

In [ ]:
%%time
vds = hl.vds.filter_intervals(vds, [locus_interval])

In [ ]:
%%time
vds.variant_data.count()

# count() (971624, 535662)

The field sample_id contains the person_ids, so we set the field sample_id as key while reading the table.

In [ ]:
flagged_samples = hl.import_table(flagged_samples_path, key = "s")

In [ ]:
vds = hl.vds.filter_samples(vds, flagged_samples, keep = False, remove_dead_alleles = True)

In [ ]:
# %%time
# vds.variant_data.count()

In [ ]:
vds.variant_data.describe()

In [ ]:
%%time
mt = hl.vds.to_dense_mt(vds)
mt.write(f'{bucket}/hail_files/lmna_1mb.mt', overwrite=True)